In [5]:
import pandas as pd
import warnings


warnings.filterwarnings('ignore')

train_df = pd.read_csv('./ratings_train.txt', sep='\t', encoding='utf8')
train_df.head(5)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [7]:
train_df['label'].value_counts()

label
0    75173
1    74827
Name: count, dtype: int64

In [9]:
train_df.isnull().sum() # document null 값 5개 존재

id          0
document    5
label       0
dtype: int64

In [8]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [10]:
import re

train_df = train_df.fillna(' ') # null 값 5개에 대해 공백으로 치환

# 정규 표현식을 이용하여 숫자를 공백으로 변경 (정규 표현식으로 \d 는 숫자를 의미함)
train_df['document'] = train_df['document'].apply(lambda x : re.sub(r'\d+', ' ', x))

test_df = pd.read_csv('./ratings_test.txt', sep='\t', encoding='utf8')
test_df = test_df.fillna(' ')
test_df['document'] = test_df['document'].apply(lambda x : re.sub(r'\d+', ' ', x))

# id 컬럼 삭제
train_df.drop('id', axis=1, inplace=True)
test_df.drop('id', axis=1, inplace=True)


In [18]:
from konlpy.tag import Okt

okt = Okt()
def okt_tokenizer(text : str) -> list:
    return okt.morphs(text)

okt_tokenizer('아버지가방에 들어가신다')

['아버지', '가방', '에', '들어가신다']

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

tfidf_vect = TfidfVectorizer(tokenizer=okt_tokenizer, ngram_range=(1, 2), min_df=3, max_df=0.9)
tfidf_vect.fit(train_df['document'])
tfidf_matrix_train = tfidf_vect.transform(train_df['document'])

In [20]:
tfidf_matrix_train.shape

(150000, 129276)

In [21]:
# Logistic Regression 을 이용하여 감성 분석 Classification 수행

lg_clf = LogisticRegression(random_state=0, solver='liblinear')

# Parameter C 최적화를 위해 GridSearchCV 를 이용
params = { 'C' : [1, 3.5, 4.5, 5.5, 10] }
grid_cv = GridSearchCV(lg_clf, param_grid=params, scoring='accuracy', verbose=1)
grid_cv.fit(tfidf_matrix_train, train_df['label'])
print(grid_cv.best_params_, round(grid_cv.best_score_, 4))




Fitting 5 folds for each of 5 candidates, totalling 25 fits
{'C': 3.5} 0.8628


In [25]:
from sklearn.metrics import accuracy_score

# 학습 데이터를 적용한 TfidVectorize 를 이용하여 테스트 데이터를 TF-IDF 값으로 Feature 변환함
tfidf_matrix_test = tfidf_vect.transform(test_df['document'])

# GridSearch 에서 best 모델을 들고옵니다.
best_estimator = grid_cv.best_estimator_
preds = best_estimator.predict(tfidf_matrix_test)

print('Logistic Regression 정확도:',accuracy_score(test_df['label'], preds))

Logistic Regression 정확도: 0.86172
